## Libraries

In [ ]:
%pip install opencv-python

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models

## Data Loading and Preprocessing

In [ ]:
IMG_SIZE = 128

def load_data(image_dir, mask_dir):
    images = []
    masks = []

    img_files = sorted(os.listdir(image_dir))

    for img_name in img_files:
        img_path = os.path.join(image_dir, img_name)
        mask_name = img_name.replace(".jpg", ".png")
        mask_path = os.path.join(mask_dir, mask_name)

        # Load image
        img = cv2.imread(img_path)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0

        # Load mask
        mask = cv2.imread(mask_path, 0)
        mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
        mask = (mask > 0).astype(np.float32)

        images.append(img)
        masks.append(mask)

    images = np.array(images)
    masks = np.array(masks)

    # Add channel to mask
    masks = np.expand_dims(masks, axis=-1)

    return images, masks

In [ ]:
X_train, y_train = load_data(
    "./data/segmentation_task/train/images/",
    "./data/segmentation_task/train/masks/"
)

## U-Net Architecture

In [ ]:
def unet_model():
    inputs = layers.Input((128, 128, 3))

    # Encoder
    c1 = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(32, 3, activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(64, 3, activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(64, 3, activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D()(c2)

    # Bottleneck
    c3 = layers.Conv2D(128, 3, activation='relu', padding='same')(p2)

    # Decoder
    u1 = layers.UpSampling2D()(c3)
    u1 = layers.concatenate([u1, c2])
    c4 = layers.Conv2D(64, 3, activation='relu', padding='same')(u1)

    u2 = layers.UpSampling2D()(c4)
    u2 = layers.concatenate([u2, c1])
    c5 = layers.Conv2D(32, 3, activation='relu', padding='same')(u2)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c5)

    model = models.Model(inputs, outputs)
    return model

## Dice Loss

In [ ]:
import tensorflow as tf

def dice_loss(y_true, y_pred):
    smooth = 1e-6
    intersection = tf.reduce_sum(y_true * y_pred)
    return 1 - (2. * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)


# Dice loss is a loss function used primarily for semantic image segmentation
# to measure the overlap between predicted segmentation and ground truth

## Compile Model

In [ ]:
model = unet_model()

model.compile(
    optimizer='adam',
    loss=lambda y_true, y_pred: tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred),
    metrics=['accuracy']
)

## Train Model

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=8
)

## Prediction

In [ ]:
pred = model.predict(X_train[0:1])[0]
pred_bin = pred > 0.3

## Visualization

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(X_train[0])
plt.title("Original")

plt.subplot(1,3,2)
plt.imshow(y_train[0].squeeze(), cmap='gray')
plt.title("Ground Truth")

plt.subplot(1,3,3)
plt.imshow(pred_bin.squeeze(), cmap='gray')
plt.title("Prediction")
plt.show()

## Save Model V1

In [ ]:
model.save("./model/unet_model_v1.h5")

# **Improvement**

## Better Loss Weighting

In [ ]:
model = unet_model()

model.compile(
    optimizer='adam',
    loss = lambda y_true, y_pred: 0.5 * tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred),
    metrics=['accuracy']
)

## Increasing epoch number to 40

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=8
)

In [ ]:
pred = model.predict(X_train[0:1])[0]
pred_bin = pred > 0.3

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(X_train[0])
plt.title("Original")

plt.subplot(1,3,2)
plt.imshow(y_train[0].squeeze(), cmap='gray')
plt.title("Ground Truth")

plt.subplot(1,3,3)
plt.imshow(pred_bin.squeeze(), cmap='gray')
plt.title("Prediction")

## Save Model V2

In [ ]:
model.save("./model/unet_model_v2.h5")

## Model Imrovement With BatchNorm and Diceloss

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    return x


def unet_model():
    inputs = layers.Input((128, 128, 3))

    # Encoder
    c1 = conv_block(inputs, 64)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 128)
    p2 = layers.MaxPooling2D()(c2)

    # Bottleneck
    c3 = conv_block(p2, 256)

    # Decoder
    u1 = layers.UpSampling2D()(c3)
    u1 = layers.concatenate([u1, c2])
    c4 = conv_block(u1, 128)

    u2 = layers.UpSampling2D()(c4)
    u2 = layers.concatenate([u2, c1])
    c5 = conv_block(u2, 64)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c5)

    return models.Model(inputs, outputs)

In [ ]:
def dice_loss(y_true, y_pred):
    smooth = 1e-6
    intersection = tf.reduce_sum(y_true * y_pred)
    return 1 - (2. * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)